In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import ROOT as root
from ROOT import TH2F
from ROOT import TH1F
from openpyxl import Workbook
import pytz
from datetime import datetime
import json
from ROOT import TCanvas
import statistics
import pandas as pd
#%jsroot on

Welcome to JupyROOT 6.30/04


In [2]:
n_wafer = 83 #
n_row = 6 #
n_col = 6 # 
n_sensor = 24
start_wafer = 1 #first wafer number in the dataset
start_row = 1 #the first row of the wafer to be measured
start_col = 1 #the first column of the wafer to be measured 
start_sensor = 1


dtz = datetime(2026, 5, 9, 12, 0, 0)
dtz = dtz.replace(tzinfo=pytz.utc)
dtz.astimezone(pytz.timezone("Europe/Rome"))

user = "fsiviero" # cern user of who's uploading the test
location = "CERN" # where the test was performed


wb = pd.read_excel('/Users/icosivi/Desktop/PRE-SERIE/HPK/HPK_16x16_PRE-SERIES.xlsx') # excel that maps 'serial number --> vendor, batch, wafer, row, column'

save_path = '/Users/icosivi/Desktop/PRE-SERIE/HPK/'

vendor_file ="/Users/icosivi/Desktop/PRE-SERIE/HPK/HPK_16x16_PRE-SERIES_Vendor_IV.json"
    

In [3]:
def get_category_by_component(data, component_id):
    """
    Cerca all'interno della lista di dizionari l'oggetto che ha 
    il campo 'component' corrispondente a component_id.
    """
    # Se il dato passato è una stringa (percorso file), caricalo
    if isinstance(data, str):
        try:
            with open(data, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except Exception as e:
            return f"Errore nel caricamento del file: {e}"

    # Itera sulla lista di componenti
    for entry in data:
        # Verifica se il campo 'component' esiste e coincide
        if entry.get("component") == component_id:
            # Restituisce la categoria se trovata, altrimenti un messaggio di default
            return entry.get("category", "Campo 'category' non presente")
    
    return "Componente non trovato"
  
cat = get_category_by_component("/Users/icosivi/Desktop/PRE-SERIE/HPK/HPK_16x16_PRE-SERIES_Vendor_IV.json","PRE01000000000000010110")
print(cat)

GOOD


In [8]:
optical_inspection_passed = bool()
optical_inspection_json = []

wafer_number = 83

for j in range(n_sensor):
    #if j+start_sensor != 24:
        cat = get_category_by_component(vendor_file,str(wb[(wb['Wafer'] == wafer_number) & (wb['Sensor Number'] == j+start_sensor) ]['SerialNumber'].iloc[0]))
        if str(cat) != 'BAD':
            opt = {'component': str(wb[(wb['Wafer'] == wafer_number) & (wb['Sensor Number'] == j+start_sensor) ]['SerialNumber'].iloc[0]),
                      'name': 'Optical Inspection',
                      'measurement_date': dtz.isoformat(), #year, month, day, hour, minute, second
                      'location': location,
                      'user_created': user,
                      'version': 'v0',
                      'passed': True,
                      'comment': None
                }
            optical_inspection_json.append(opt)


with open(save_path+"HPK_CERN_optical_inspection_W"+str(wafer_number)+".json", 'w') as f:
   json.dump(optical_inspection_json, f, indent=4)